<!-- NB_DOC_INTRO_V1 -->
# Analyse de survie (Survival Analysis)

> 📚 **Doc thematique** : [docs/03_ML.md](docs/03_ML.md) (Machine Learning classique)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**L'analyse de survie** modelise le **temps avant un evenement** (panne machine, churn client, deces patient, conversion). Le defi specifique : on doit gerer la **censure** (pour beaucoup d'observations, on ne sait pas QUAND l'evenement va se produire — juste qu'a `t`, il n'avait pas encore eu lieu).

Un modele de regression classique sur la duree :
- **Ignore** les cas censures → perte d'information massive (souvent 50%+ des donnees)
- **Ou** les traite comme "duree = date_fin" → biais grave (sous-estime la duree de vie)

L'analyse de survie modelise correctement cette incertitude via la **fonction de survie** `S(t) = P(T > t)` et le **hazard** `h(t)` (risque instantane).

Ce notebook couvre les 4 familles de modeles avec **code execute** sur des datasets reels :
- **Non-parametrique** : Kaplan-Meier (estimer S(t) sans hypothese)
- **Semi-parametrique** : Cox Proportional Hazards (le standard)
- **Parametrique** : Weibull AFT, log-normal AFT
- **ML** : Random Survival Forest, Gradient Boosting Survival

Cas d'usage : maintenance predictive (RUL), churn, credit risk, sante.

Versions : `lifelines >= 0.27`, `scikit-survival >= 0.22`.

## Plan

1. Installation et concepts (censure, hazard, survie)
2. **Kaplan-Meier** non-parametrique + log-rank test
3. **Cox Proportional Hazards** + verification hypotheses
4. **AFT models** (Weibull, log-normal)
5. **Random Survival Forest** (sksurv)
6. **Metriques** : C-index, Brier, IBS, time-dependent AUC
7. **Comparatif** des modeles
8. **Cas industriel** : turbofan RUL (idee)
9. Pieges et anti-patterns
10. References


## 1. Installation et concepts cles

### Vocabulaire

| Notion | Definition |
|---|---|
| **T** | Variable aleatoire = temps jusqu'a l'evenement |
| **S(t) = P(T > t)** | **Fonction de survie** — proba de "survivre" au-dela de `t` |
| **F(t) = 1 - S(t)** | Fonction de repartition (CDF) |
| **h(t)** | **Hazard** — taux instantane d'evenement a `t`, sachant survie jusque-la |
| **H(t) = ∫_0^t h(u) du** | Cumulative hazard |
| **Censure droite** | T > date d'observation (le plus courant) |
| **Censure gauche** | T < date d'observation |
| **Censure par intervalle** | T ∈ [a, b] |
| **Troncature** | Sujet n'entre dans l'etude qu'apres un certain temps |

### Relation fondamentale
`S(t) = exp(-H(t))` (le **theoreme fondamental** de la survie).


In [ ]:
# pip install -q lifelines scikit-survival numpy pandas matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

import lifelines
print(f"lifelines version : {lifelines.__version__}")

SEED = 42
np.random.seed(SEED)

## 2. Kaplan-Meier — estimateur non-parametrique de S(t)

Le **Kaplan-Meier** estime S(t) sans hypothese sur la forme. Calcul recursif :
`S(t_i) = S(t_{i-1}) × (1 - d_i / n_i)` ou `d_i` = nb deces a t_i, `n_i` = nb a risque.

C'est **toujours** la premiere chose a tracer dans un projet survie : ca donne l'allure de la distribution.


In [ ]:
from lifelines import KaplanMeierFitter
from lifelines.datasets import load_waltons

# Dataset Waltons : durees + groupe (miR-137 vs control)
df = load_waltons()
print(df.head())
print(f"\nShape : {df.shape}, evenements observes : {df['E'].sum()} / {len(df)}")

In [ ]:
# === KM global ===
kmf = KaplanMeierFitter()
kmf.fit(durations=df["T"], event_observed=df["E"], label="Tous")

fig, ax = plt.subplots(figsize=(10, 5))
kmf.plot_survival_function(ax=ax)
ax.set(xlabel="Temps", ylabel="P(survie)", title="Kaplan-Meier — population entiere")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Mediane de survie
print(f"Mediane de survie : {kmf.median_survival_time_}")

### 2.1 KM par groupe + log-rank test

**Log-rank test** : H0 = les 2 groupes ont la **meme** distribution de survie.


In [ ]:
from lifelines.statistics import logrank_test

# KM par groupe
fig, ax = plt.subplots(figsize=(10, 5))
for name, sub in df.groupby("group"):
    KaplanMeierFitter().fit(sub["T"], sub["E"], label=name).plot_survival_function(ax=ax)
ax.set(xlabel="Temps", ylabel="P(survie)", title="Kaplan-Meier par groupe")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Log-rank
A = df[df["group"] == "miR-137"]
B = df[df["group"] == "control"]
res = logrank_test(A["T"], B["T"], A["E"], B["E"])
print(f"Log-rank stat : {res.test_statistic:.4f}, p-value : {res.p_value:.6f}")
print(f"=> Difference significative : {'OUI' if res.p_value < 0.05 else 'NON'}")

## 3. Cox Proportional Hazards — semi-parametrique

**Le standard** en survie. Modele : `h(t | x) = h_0(t) × exp(β·x)`.
- `h_0(t)` = **baseline hazard** (non-parametre, estime non-parametriquement)
- `β` = coefficients (**parametres**)
- **Hypothese cle** : les hazards sont **proportionnels** entre individus (le ratio ne depend pas de t)

Le `exp(β_i)` se lit comme un **hazard ratio** : `HR > 1` = facteur de risque accru, `HR < 1` = facteur protecteur.


In [ ]:
from lifelines import CoxPHFitter
from lifelines.datasets import load_rossi

# Rossi : recidivisme apres prison (semaines avant arrestation), arrest=1 si arrest, 0 si censure
data = load_rossi()
print(data.head())
print(f"\nColumns : {data.columns.tolist()}")
print(f"Events : {data['arrest'].sum()} / {len(data)}")

In [ ]:
cph = CoxPHFitter()
cph.fit(data, duration_col="week", event_col="arrest")
cph.print_summary()

**Lecture du summary** :
- `exp(coef)` = hazard ratio. Ex: `fin=-0.38` → `HR=0.68` → l'aide financiere reduit le risque d'arrestation de 32%.
- `p` < 0.05 → coef significatif
- `c-index` global (`concordance`) : qualite predictive


In [ ]:
# === Verifier l'hypothese de proportionnalite ===
results = cph.check_assumptions(data, p_value_threshold=0.05, show_plots=False)
# Si une variable echoue, options :
# - stratifier sur cette variable (cph.fit(..., strata=['age']))
# - ajouter une interaction avec t (variable a effet variable dans le temps)
# - utiliser AFT au lieu de Cox

In [ ]:
# === Prediction de la fonction de survie pour 5 nouveaux individus ===
fig, ax = plt.subplots(figsize=(10, 5))
cph.predict_survival_function(data.iloc[:5]).plot(ax=ax)
ax.set(xlabel="Semaines", ylabel="P(pas d'arrestation)",
       title="Survival functions predites par Cox (5 premiers individus)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. AFT models — parametriques (Weibull, log-normal)

**AFT** = **Accelerated Failure Time**. Au lieu de modeliser le hazard, on modelise directement `log(T) = β·x + ε`. Les coefs s'interpretent comme un **facteur d'acceleration** du temps de survie.

- **Weibull** : flexible (hazard croissant/decroissant/constant selon shape)
- **Log-normal** : T suit une log-normale, bon pour duree de vie batterie
- **Log-logistic** : hazard d'abord croissant puis decroissant


In [ ]:
from lifelines import WeibullAFTFitter, LogNormalAFTFitter

# Weibull AFT
weibull = WeibullAFTFitter()
weibull.fit(data, duration_col="week", event_col="arrest")
print("=== Weibull AFT ===")
weibull.print_summary(decimals=3)

In [ ]:
# Log-normal AFT (parfois meilleur sur duree de vie longue)
lognorm = LogNormalAFTFitter()
lognorm.fit(data, duration_col="week", event_col="arrest")
print("=== Log-normal AFT ===")
print(f"AIC : {lognorm.AIC_:.2f}  vs Weibull AIC : {weibull.AIC_:.2f}  vs Cox AIC : N/A (semi-param)")

## 5. Random Survival Forest (scikit-survival)

Approche ML : pas d'hypothese de proportionnalite, capture les non-linearites et interactions automatiquement.


In [ ]:
# pip install -q scikit-survival
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, integrated_brier_score
from sksurv.datasets import load_veterans_lung_cancer

# Dataset : Veterans lung cancer (Kalbfleisch & Prentice 1980)
X_vlc, y_vlc = load_veterans_lung_cancer()
# y_vlc est un structured array : (Status, Survival_in_days)
X_vlc_num = X_vlc.select_dtypes(include=[np.number])  # garder que num pour la demo simple
print(f"Shape X : {X_vlc_num.shape}")
print(f"y dtype : {y_vlc.dtype}")
print(f"Events : {y_vlc['Status'].sum()} / {len(y_vlc)}")

In [ ]:
# Train RSF
from sklearn.model_selection import train_test_split

# Le split doit preserver la structure des y
indices = np.arange(len(X_vlc_num))
idx_tr, idx_te = train_test_split(indices, test_size=0.3, random_state=SEED)

X_tr, X_te = X_vlc_num.iloc[idx_tr], X_vlc_num.iloc[idx_te]
y_tr, y_te = y_vlc[idx_tr], y_vlc[idx_te]

rsf = RandomSurvivalForest(n_estimators=100, min_samples_split=10,
                           min_samples_leaf=15, random_state=SEED, n_jobs=-1)
rsf.fit(X_tr, y_tr)

# Score = C-index sur le test (sksurv le calcule via .score())
c_index_train = rsf.score(X_tr, y_tr)
c_index_test = rsf.score(X_te, y_te)
print(f"C-index train : {c_index_train:.4f}  (optimistic)")
print(f"C-index test  : {c_index_test:.4f}")
print(f"  -> 0.5 = random, 0.7 = utilisable, 0.8+ = excellent")

In [ ]:
# === Gradient Boosting Survival (souvent meilleur que RSF) ===
gbsa = GradientBoostingSurvivalAnalysis(n_estimators=100, learning_rate=0.1,
                                         max_depth=3, random_state=SEED)
gbsa.fit(X_tr, y_tr)
print(f"GBSA C-index test : {gbsa.score(X_te, y_te):.4f}")

## 6. Metriques de survie

### 6.1 C-index (Concordance Index)

Probabilite qu'une paire (i, j) avec `t_i < t_j` ait `risk_i > risk_j` (uniquement chez les non-censures).
- 0.5 = random
- 0.7 = utilisable
- 0.8+ = excellent


In [ ]:
from sksurv.metrics import concordance_index_censored

# Predictions de risque
risk_test = rsf.predict(X_te)
c_index, concordant, discordant, tied_risk, tied_time = concordance_index_censored(
    y_te["Status"], y_te["Survival_in_days"], risk_test,
)
print(f"C-index : {c_index:.4f}")
print(f"  Concordant pairs : {concordant}")
print(f"  Discordant pairs : {discordant}")
print(f"  Ratio : {concordant / (concordant + discordant):.4f}")

### 6.2 Brier score et IBS (Integrated Brier Score)

Le Brier score `BS(t)` mesure l'erreur quadratique de la prediction `S(t)` a un temps donne.
**IBS** = integration de BS sur un intervalle. Plus bas = mieux. < 0.25 = bon.


In [ ]:
# Predire les fonctions de survie pour chaque test
from sksurv.functions import StepFunction

# Time grid evaluation (eviter les bords)
lower, upper = np.percentile(y_te["Survival_in_days"], [10, 90])
times = np.arange(int(lower), int(upper), 5)

# RSF : .predict_survival_function -> liste de StepFunction
surv_funcs = rsf.predict_survival_function(X_te)
surv_at_times = np.asarray([[fn(t) for t in times] for fn in surv_funcs])

# IBS attend (y_train, y_test, estimates, times)
try:
    ibs = integrated_brier_score(y_tr, y_te, surv_at_times, times)
    print(f"Integrated Brier Score : {ibs:.4f}  (plus bas = mieux, < 0.25 = bon)")
except Exception as e:
    print(f"IBS calc : {e}")

## 7. Comparatif des modeles testes


In [ ]:
# Comparer les C-indexes
results = []

# Kaplan-Meier comme baseline (pas de features)
from lifelines.utils import concordance_index
results.append(("KM (baseline)", 0.5))   # KM seul ne predit pas, c-index = 0.5

# Cox sur Rossi (dataset different — illustratif)
results.append(("Cox (Veterans)", c_index_test))

# RSF
results.append(("RSF (Veterans)", rsf.score(X_te, y_te)))

# GBSA
results.append(("GBSA (Veterans)", gbsa.score(X_te, y_te)))

pd.DataFrame(results, columns=["Model", "C-index test"])

## 8. Cas industriel — Remaining Useful Life (RUL)

Cas classique : turbofan engine (NASA C-MAPSS). Pour chaque moteur, capteurs lus a chaque cycle. La cible : `RUL = cycles_restants` au moment de la mesure.

C'est un probleme de **survie multi-capteurs avec covariates variant dans le temps**. Approches courantes :
1. **Cox time-varying** (Cox avec covariates evoluant)
2. **DeepSurv** (NN avec loss de Cox)
3. **Random Survival Forest** sur features fenetrees
4. **Multi-task regression** : predire RUL directement, censure handled via custom loss

Voir [TS_Maintenance_Predictive_GOOD.ipynb](./TS_Maintenance_Predictive_GOOD.ipynb) pour le dataset C-MAPSS complet (non duplicate ici).


## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| Traiter T comme regression directe | Utiliser un modele de survie (gere censure) |
| Traiter evenement binaire (E) seul | On perd l'info temporelle de la duree |
| Cox sans verifier proportionnalite | `check_assumptions()` ; sinon stratifier ou AFT |
| Confondre **survival** (S(t)) et **hazard** (h(t)) | S decroit ; h peut croitre/decroitre |
| Confondre `T = duree` et `T = date_evenement` | Toujours **duree** (= temps depuis entree dans l'etude) |
| Splitter par individu sur donnees longitudinales | Splitter en preservant la cluster ID (GroupKFold) |
| Metrique = MSE sur T | C-index (ou IBS) |
| Ignorer la censure de gauche / par intervalle | Modeles dedies (Cox interval-censored, NPMLE) |
| Comparer 2 modeles sur 2 datasets differents | Comparer sur le meme split CV |
| Oublier que **immortal time bias** existe | Definir t=0 clairement (date d'entree dans le suivi) |


## 10. References

### Documentation officielle
- **lifelines** : https://lifelines.readthedocs.io/
- **scikit-survival** : https://scikit-survival.readthedocs.io/
- **pycox** (DeepSurv, DeepHit) : https://github.com/havakv/pycox

### Livres
- Klein & Moeschberger (2003). *Survival Analysis: Techniques for Censored and Truncated Data*. Standard pedagogique.
- Therneau & Grambsch (2000). *Modeling Survival Data: Extending the Cox Model*. Reference Cox.

### Papers
- Cox (1972). *Regression Models and Life-Tables*. Le papier fondateur.
- Ishwaran et al. (2008). *Random Survival Forests*.
- Katzman et al. (2018). *DeepSurv: Personalized Treatment Recommender System Using Cox PH Deep Neural Network*.

### Voir aussi (collection)
- [DS_Metrics_Reference.ipynb](./DS_Metrics_Reference.ipynb) — section "Survival" (C-index, IBS)
- [TS_Maintenance_Predictive_GOOD.ipynb](./TS_Maintenance_Predictive_GOOD.ipynb) — RUL sur NASA C-MAPSS
- [DS_Causal_Inference.ipynb](./DS_Causal_Inference.ipynb) — effets causaux sur duree de survie (DiD, IV)
